# 01 · Document ingestion

This experiment reads local HTML, extracts sections, creates LlamaIndex `Document` objects,
and splits them into nodes. No embeddings, index, vector database, LLM, or API key is involved.
Run cells from top to bottom after installing `pip install -e ".[dev,ingestion]"`.
The tracked notebook deliberately has no saved outputs; counts and excerpts come from your run.

## 1. Why these sources?

| Official source | Role | Limits |
| --- | --- | --- |
| [General fishing regulations](https://www.ontario.ca/document/ontario-fishing-regulations-summary/general-fishing-regulations) | Provincial regulatory context | A summary, not a complete set of legal instruments |
| [Fisheries Management Zone 17](https://www.ontario.ca/document/ontario-fishing-regulations-summary/fisheries-management-zone-17) | Local seasons, limits, exceptions and sanctuaries | Do not detach zone-wide rules from exceptions or infer a lake boundary from this text alone |
| [Learn to Fish – Guide](https://www.ontario.ca/page/learn-fish-guide) | Government fundamentals guide with a dedicated **Smallmouth and largemouth bass** section | Broad Ontario advice, not a bass-only book or lake survey, regulatory advice in this older guide is not current legal authority |

The first two chapters belong to the **2026** summary (effective January 1, 2026),
verified on **November 1, 2026** through the official summary landing page and linked PDF.
Their original web publication year is 2019, which should not be confused with the regulation year.
The guide was published in 2016 and updated in 2023.

`data/sources.json` holds provenance, exact-byte SHA-256 hashes, and the usage review.
[Ontario's copyright policy](https://www.ontario.ca/page/copyright-information) permits unchanged
noncommercial copies with attribution and Crown copyright acknowledgment, unless otherwise indicated.
The original HTML files are stored unchanged under Git-ignored `data/documents/`.
Extraction and chunk inspection are local research transformations, not official reproductions.

In [ ]:
import hashlib
import json
from collections import Counter
from importlib.metadata import version
from pathlib import Path

from bs4 import BeautifulSoup
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.utils import get_tokenizer

# Jupyter commonly starts in either the repository root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "data/sources.json").is_file(), (
    "Start Jupyter in the FishRAG root or notebooks/."
)
manifest = json.loads((ROOT / "data/sources.json").read_text(encoding="utf-8"))
sources = manifest["sources"]
assert len(sources) == 3
print({name: version(name) for name in ("llama-index-core", "beautifulsoup4")})
for source in sources:
    print(source["title"], source["access_date"], source["attribution"])

## 2. Verify the snapshots before extraction

This notebook makes no network requests. Follow the README download instructions if files
are missing. Hash mismatches stop the run: a changed official page needs a fresh review
and manifest update, rather than silently inheriting an old access date or regulation year.
HTML has no stable page numbers, so `page` is `None`; headings provide section locators.

In [ ]:
soups = {}
for source in sources:
    path = ROOT / source["local_path"]
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. See README: local source snapshots.")
    raw = path.read_bytes()
    if hashlib.sha256(raw).hexdigest() != source["sha256"]:
        raise ValueError(
            f"Snapshot changed: {path.name}. Review source and update manifest."
        )
    soup = BeautifulSoup(raw.decode("utf-8"), "html.parser")
    main = soup.find("main")
    heading = main.find("h1") if main else None
    assert heading and heading.get_text(" ", strip=True) == source["title"]
    body = main.select_one(".body-field")
    assert body is not None, f"Content selector changed: {source['url']}"
    soups[source["source_id"]] = soup
    print(path.name, len(raw), "bytes; title and hash verified")

## 3. Extract sections, then wrap text in LlamaIndex Documents

Beautiful Soup reads the HTML body; `Document` is LlamaIndex's text-plus-metadata container.
Splitting at headings first keeps each chunk inside one identifiable section. We keep
paragraphs, lists, glossary definitions, and tables, and exclude navigation and image data.
Heading paths preserve context such as **Zone-wide seasons and limits → bass** versus exceptions.

`content_kind` is an explicit **section-level editorial label**, not a model prediction or
claim-level scientific validation. Regulation summaries stay separate from guide advice.
The bass section mixes reported biological facts and technique advice, so we label it
`mixed_biology_and_technique` instead of pretending every sentence has one type.
Guide sections about rules are labeled `regulatory_guidance`, not current regulations.
Other species and general sections remain available as context rather than being mislabeled as bass advice.

In [ ]:
def content_kind(source, heading_path):
    if source["source_type"] == "regulation_summary":
        return "regulation_summary"
    if "Rules and regulations" in heading_path:
        return "regulatory_guidance"
    if "Smallmouth and largemouth bass" in heading_path:
        return "mixed_biology_and_technique"
    if heading_path and heading_path[0] in {
        "Fishing rods, reels and line",
        "Tackle and other gear",
        "Knots and casting",
        "Tips and techniques",
    }:
        return "technique_advice"
    return "general_guide_context"


def extract_sections(body):
    # Only take outer blocks so nested lists and table cells are not duplicated.
    blocks = {"p", "ul", "ol", "dl", "table"}
    headings = {}
    lines = []
    section_anchor = None
    sections = []
    for tag in body.find_all(["h2", "h3", "h4", "h5", "h6", *sorted(blocks)]):
        if any(parent.name in blocks for parent in tag.parents if parent is not body):
            continue
        if tag.name.startswith("h"):
            if lines:
                sections.append(
                    (list(headings.values()), section_anchor, "\n\n".join(lines))
                )
            level = int(tag.name[1])
            headings = {key: value for key, value in headings.items() if key < level}
            headings[level] = tag.get_text(" ", strip=True)
            section_anchor = tag.get("id")  # Do not invent missing anchors.
            lines = []
        else:
            text = tag.get_text(" ", strip=True)
            if text:
                lines.append(text)
    if lines:
        sections.append((list(headings.values()), section_anchor, "\n\n".join(lines)))
    return sections


documents = []
for source in sources:
    body = soups[source["source_id"]].find("main").select_one(".body-field")
    for section_index, (heading_path, anchor, text) in enumerate(
        extract_sections(body)
    ):
        section = " > ".join(heading_path) or "Introduction (before first heading)"
        metadata = {
            key: source[key]
            for key in (
                "source_id",
                "title",
                "url",
                "publisher",
                "geographic_scope",
                "source_type",
                "publication_date",
                "publication_year",
                "updated_date",
                "regulation_year",
                "access_date",
                "page",
                "sha256",
                "attribution",
                "terms_url",
            )
        }
        metadata.update(
            section=section,
            section_url=f"{source['url']}#{anchor}" if anchor else source["url"],
            content_kind=content_kind(source, heading_path),
        )
        documents.append(
            Document(
                id_=f"{source['source_id']}:{section_index}",
                text=f"{section}\n\n{text}",
                metadata=metadata,
                # Retain provenance without spending the text budget on metadata.
                excluded_embed_metadata_keys=list(metadata),
                excluded_llm_metadata_keys=list(metadata),
            )
        )

assert {doc.metadata["source_id"] for doc in documents} == {
    s["source_id"] for s in sources
}
print("Extracted sections:", len(documents))
print(
    "Section labels:", dict(Counter(doc.metadata["content_kind"] for doc in documents))
)
for source in sources:
    selected = [
        doc for doc in documents if doc.metadata["source_id"] == source["source_id"]
    ]
    print(
        source["source_id"],
        "sections:",
        len(selected),
        "characters:",
        sum(len(d.text) for d in selected),
    )

In [ ]:
# Inspect one relevant section from each source, including its full provenance.
samples = [
    next(
        d
        for d in documents
        if d.metadata["source_id"] == "ontario-general-regulations"
        and d.metadata["section"] == "Catch and retain rules"
    ),
    next(
        d
        for d in documents
        if d.metadata["source_id"] == "ontario-zone-17"
        and "Largemouth and smallmouth bass combined" in d.metadata["section"]
    ),
    next(
        d
        for d in documents
        if d.metadata["source_id"] == "ontario-learn-to-fish"
        and "Smallmouth and largemouth bass" in d.metadata["section"]
    ),
]
for doc in samples:
    print(json.dumps(doc.metadata, indent=2, ensure_ascii=False))
    print(doc.text[:1600], "\n")

## 4. Chunk with SentenceSplitter

Start with a **512-token text budget** and **64-token overlap**. These are experiment
settings, not tuned values. The default LlamaIndex tokenizer is local; it does not call
an LLM or generate embeddings. A node is a smaller text unit with inherited provenance
and a reference to its parent section. Sections shorter than the budget remain short.
Overlap applies within a section; it cannot carry exceptions from a different section.

In [ ]:
tokenizer = get_tokenizer()
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=64, tokenizer=tokenizer)
nodes = splitter.get_nodes_from_documents(documents)
parents = {doc.id_: doc for doc in documents}
assert nodes and all(node.text.strip() for node in nodes)
for node in nodes:
    parent = parents[node.ref_doc_id]
    assert node.metadata == parent.metadata, "Provenance must survive chunking."
    assert node.embedding is None

print("Total sections:", len(documents), "Total chunks:", len(nodes))
for source in sources:
    selected = [n for n in nodes if n.metadata["source_id"] == source["source_id"]]
    sizes = [len(tokenizer(n.text)) for n in selected]
    print(
        source["source_id"],
        {
            "chunks": len(selected),
            "min_tokens": min(sizes),
            "max_tokens": max(sizes),
            "mean_tokens": round(sum(sizes) / len(sizes), 1),
        },
    )

for doc in samples:
    sample = next(node for node in nodes if node.ref_doc_id == doc.id_)
    print("\nSOURCE:", sample.metadata["title"])
    print("SECTION:", sample.metadata["section"])
    print("TYPE:", sample.metadata["content_kind"])
    print("URL:", sample.metadata["section_url"])
    print(sample.text[:1000])

## 5. What this experiment does not establish

- HTML snapshots are not a complete offline website. Images, maps, linked legal instruments,
  PDFs, and interactive content are not loaded. There is no OCR and no stable page number.
- Whitespace and HTML styling are normalized. Nested lists, tables and glossary definitions
  become linear text; important visual grouping and link targets can be lost. Compare samples
  against the official pages, especially exceptions and sanctuaries. A valid hash does not prove
  that every sentence was extracted correctly.
- The heading selector is specific to the current Ontario pages. Assertions catch missing
  content/title and changed bytes, but do not guarantee semantic completeness.
- Biology and technique share paragraphs in the bass guide. The mixed label is intentional;
  sentence-level separation would need careful manual annotation, not guessed facts.
- A regulation summary is not the law itself. The guide's older licence/rules advice must not
  override current regulations. Neither a zone-wide bass chunk nor a chunk count establishes
  what is legal at a particular location and date. Related exceptions need to travel with future answers.
- Character/token counts only describe extraction and chunk size. They are not retrieval,
  faithfulness, or fishing-success evaluation results. No such evaluation has been run here.

**Next experiment:** manually review section boundaries and missing context, then create a small
source-linked question set before choosing an embedding model or measuring Recall@k/MRR.
Keep extracted content local. Clear all outputs before sharing this notebook (see README).